# Sample comparison notebook

This notebook is the analysis workspace for comparing samples.

In [ ]:
from pathlib import Path
import pandas as pd

# Configure these
results_root = Path("/home/peterkad/pkadmaster/indel_scanner/results")
samples = ["tr_plus_unmapped_diploid_v2", "ph_plus_unmapped_diploid_v2"]


def latest_run_dir(sample_root: Path) -> Path | None:
    if not sample_root.exists():
        return None
    run_dirs = [p for p in sample_root.iterdir() if p.is_dir()]
    if not run_dirs:
        return None
    return max(run_dirs, key=lambda p: p.stat().st_mtime)


records = []
missing = []

for sample in samples:
    sample_root = results_root / sample
    run_dir = latest_run_dir(sample_root)
    if run_dir is None:
        missing.append((sample, "no_run_dir"))
        continue
    per_type = run_dir / "per_type_mutation_frequency.tsv"
    callable_bases = run_dir / "callable_bases.tsv"
    if not per_type.exists() or not callable_bases.exists():
        missing.append((sample, str(run_dir)))
        continue
    records.append(
        {
            "sample": sample,
            "run_dir": run_dir,
            "per_type": per_type,
            "callable_bases": callable_bases,
        }
    )

pd.DataFrame(records), pd.DataFrame(missing, columns=["sample", "issue"])

In [ ]:
def load_per_type(path: Path, sample: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    df["sample"] = sample
    drop_cols = ["unique_rate", "unique_sites", "str_region_count"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return df


def load_callable(path: Path, sample: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    df["sample"] = sample
    drop_cols = ["str_region_count"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return df


per_type_dfs = []
callable_dfs = []

for rec in records:
    sample = rec["sample"]
    per_type_dfs.append(load_per_type(rec["per_type"], sample))
    callable_dfs.append(load_callable(rec["callable_bases"], sample))

per_type_all = pd.concat(per_type_dfs, ignore_index=True) if per_type_dfs else pd.DataFrame()
callable_all = pd.concat(callable_dfs, ignore_index=True) if callable_dfs else pd.DataFrame()

per_type_all.head()

In [ ]:
# Compare frequency by sample and STR class
comparison = (
    per_type_all.groupby(["sample", "str_class"], as_index=False)[["count", "callable_bases"]]
    .sum()
)
comparison["frequency"] = comparison["count"] / comparison["callable_bases"]

comparison = comparison.sort_values(["str_class", "sample"]).copy()
comparison["frequency"] = comparison["frequency"].map(lambda v: f"{v:.2e}" if pd.notna(v) else "NA")
comparison["callable_bases"] = comparison["callable_bases"].map(lambda v: f"{v:.0f}" if pd.notna(v) else "NA")
comparison

In [ ]:
# Compare per-mutation-type rates across samples (raw table)
per_call = per_type_all.copy()
per_call["frequency"] = per_call["count"] / per_call["callable_bases"]

per_call = per_call.sort_values(["str_class", "mutation_type", "sample"]).copy()
per_call["frequency"] = per_call["frequency"].map(lambda v: f"{v:.2e}" if pd.notna(v) else "NA")
per_call["callable_bases"] = per_call["callable_bases"].map(lambda v: f"{v:.0f}" if pd.notna(v) else "NA")

per_call[["mutation_type", "str_class", "sample", "count", "callable_bases", "frequency"]]

In [ ]:
# Split comparisons by STR vs non-STR and by size (1-10 vs >10)
per_call_num = per_type_all.copy()
per_call_num["frequency"] = per_call_num["count"] / per_call_num["callable_bases"]
per_call_num["size_bp"] = (
    per_call_num["mutation_type"].str.extract(r"_(\d+)bp")[0].astype(float)
)

per_call_num = per_call_num.dropna(subset=["size_bp"])
per_call_num["size_bp"] = per_call_num["size_bp"].astype(int)


def build_comparison(df: pd.DataFrame) -> pd.DataFrame:
    wide_freq = df.pivot_table(
        index=["mutation_type", "size_bp"],
        columns="sample",
        values="frequency",
        aggfunc="first",
    )
    wide_count = df.pivot_table(
        index=["mutation_type", "size_bp"],
        columns="sample",
        values="count",
        aggfunc="first",
    )
    wide_callable = df.pivot_table(
        index=["mutation_type", "size_bp"],
        columns="sample",
        values="callable_bases",
        aggfunc="first",
    )

    freq_cols = [f"{c}_freq" for c in wide_freq.columns]
    count_cols = [f"{c}_count" for c in wide_count.columns]
    callable_cols = [f"{c}_callable_mbp" for c in wide_callable.columns]

    wide = pd.concat(
        [
            wide_freq.set_axis(freq_cols, axis=1),
            wide_count.set_axis(count_cols, axis=1),
            wide_callable.set_axis(callable_cols, axis=1),
        ],
        axis=1,
    ).reset_index()

    wide = wide.sort_values(["size_bp", "mutation_type"]).drop(columns=["size_bp"])
    return wide


def format_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if col.endswith("_freq"):
            out[col] = out[col].map(lambda v: f"{v:.2e}" if pd.notna(v) else "NA")
        elif col.endswith("_count"):
            out[col] = out[col].map(lambda v: f"{int(v):d}" if pd.notna(v) else "NA")
        elif col.endswith("_callable_mbp"):
            out[col] = out[col].map(lambda v: f"{v / 1_000_000:.2f}" if pd.notna(v) else "NA")
    return out

# STR (1-10 bp)
str_1_10 = per_call_num[
    (per_call_num["str_class"] == "STR_motif") & (per_call_num["size_bp"] <= 10)
]
format_table(build_comparison(str_1_10))

In [ ]:
# STR (>10 bp)
str_gt_10 = per_call_num[
    (per_call_num["str_class"] == "STR_motif") & (per_call_num["size_bp"] > 10)
]
format_table(build_comparison(str_gt_10))

In [ ]:
# non-STR (1-10 bp)
non_str_1_10 = per_call_num[
    (per_call_num["str_class"] == "non_STR") & (per_call_num["size_bp"] <= 10)
]
format_table(build_comparison(non_str_1_10))

In [ ]:
# non-STR (>10 bp)
non_str_gt_10 = per_call_num[
    (per_call_num["str_class"] == "non_STR") & (per_call_num["size_bp"] > 10)
]
format_table(build_comparison(non_str_gt_10))